# IONIS Model API (Preview)

Query live IONIS predictions via the inference API.

**Note:** This notebook is a preview of upcoming functionality. The live inference API is under development.

**What you'll learn:**
- How to query the IONIS prediction API
- Interpret model predictions
- Compare predictions to historical data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from datetime import datetime, timezone

from ionis_jupyter import grid_to_latlon, grid_distance, solar_elevation

plt.rcParams["figure.figsize"] = (12, 6)

## About the IONIS Model

**IONIS** (Ionospheric Neural Inference System) predicts HF propagation using:

- **Training data:** 175M+ signatures from WSPR, RBN, Contest, DXpedition
- **Architecture:** IonisGate neural network with constrained sidecars
- **Physics integration:** PhysicsOverrideLayer for known impossible paths

**Model outputs:**
- Predicted SNR in dB (standardized, σ units relative to training mean)
- Confidence based on training data density
- Physics override status (if applicable)

In [ ]:
# API Configuration (placeholder - update when API is live)
API_BASE = "https://api.ionis-ai.com/v1"  # Future endpoint

def predict_path(tx_grid, rx_grid, band, hour, month, sfi=150, kp=2):
    """
    Query IONIS API for path prediction.
    
    Note: This is a placeholder. API not yet available.
    Returns mock data for demonstration.
    """
    # Mock prediction based on simple heuristics
    # Real API will use the trained model
    
    distance = grid_distance(tx_grid, rx_grid)
    
    # Very simplified mock logic
    base_snr = -5  # Start slightly negative
    
    # Band effect (high bands need high SFI)
    if band >= 109:  # 15m, 10m
        base_snr += (sfi - 100) / 30
    
    # Distance penalty
    base_snr -= distance / 5000
    
    # Daytime bonus for high bands
    if 10 <= hour <= 20 and band >= 107:
        base_snr += 3
    
    # Storm penalty
    base_snr -= kp * 0.5
    
    return {
        "tx_grid": tx_grid,
        "rx_grid": rx_grid,
        "band": band,
        "hour": hour,
        "month": month,
        "predicted_snr": round(base_snr, 2),
        "confidence": 0.85,  # Mock confidence
        "physics_override": False,
        "status": "mock_prediction",
    }

print("Note: Using mock predictions. Live API coming soon.")

In [ ]:
# Example prediction
result = predict_path(
    tx_grid="DN13",  # Idaho
    rx_grid="JN48",  # Central Europe
    band=107,         # 20m
    hour=14,          # 14:00 UTC
    month=3,          # March
    sfi=150,          # Moderate solar activity
    kp=2,             # Quiet geomagnetic conditions
)

print("Prediction Result:")
for key, value in result.items():
    print(f"  {key}: {value}")

In [ ]:
# Hour-by-hour prediction
tx_grid = "DN13"
rx_grid = "JN48"
band = 107
month = 3

predictions = []
for hour in range(24):
    pred = predict_path(tx_grid, rx_grid, band, hour, month)
    predictions.append(pred)

pred_df = pd.DataFrame(predictions)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(pred_df["hour"], pred_df["predicted_snr"], color="steelblue", alpha=0.7)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Predicted SNR (dB)")
ax.set_title(f"IONIS Prediction: {tx_grid} → {rx_grid} on 20m (March)")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

print("\nNote: These are mock predictions for demonstration.")

In [ ]:
# Multi-band comparison
BAND_NAMES = {105: "40m", 107: "20m", 109: "15m", 111: "10m"}

fig, ax = plt.subplots(figsize=(14, 6))

for band in BAND_NAMES.keys():
    hour_preds = []
    for hour in range(24):
        pred = predict_path(tx_grid, rx_grid, band, hour, month)
        hour_preds.append(pred["predicted_snr"])
    ax.plot(range(24), hour_preds, marker="o", label=BAND_NAMES[band], linewidth=2)

ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Predicted SNR (dB)")
ax.set_title(f"IONIS Multi-Band Prediction: {tx_grid} → {rx_grid} (March)")
ax.set_xticks(range(0, 24, 2))
ax.legend(title="Band")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Coming Soon

The live IONIS API will provide:

- **Real-time predictions** based on current solar conditions
- **Path recommendations** for specific targets
- **Band opening alerts** when conditions favor difficult paths
- **Batch predictions** for contest planning
- **Historical validation** comparing predictions to actual spots

**Stay updated:**
- GitHub: https://github.com/IONIS-AI
- Website: https://ionis-ai.com (coming soon)

## Using Historical Data Now

While waiting for the live API, use the historical datasets:

```python
from ionis_jupyter import load_dataset

# Find historical signatures for your path
wspr = load_dataset("wspr")
path_data = wspr[
    (wspr["tx_grid_4"] == "DN13") & 
    (wspr["rx_grid_4"] == "JN48") &
    (wspr["band"] == 107)
]
print(f"Historical SNR: {path_data['median_snr'].mean():.1f} dB")
```

Historical data is the ground truth the model was trained on — it tells you what actually happened on your path.